In [ ]:
# Authenticate with GCP services and Gmail
# Use Buckets and call MCP server hosted on GCP
# Use Gmail tools so LLM can send emails.

In [ ]:
# 
# AUTHENTICATED ACCESS TO PRIVATE GCS BUCKETS
# 

import sys
from google.cloud import storage
from google.oauth2 import service_account
import os


PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"] 
SERVICE_ACCOUNT_KEY_PATH = os.environ["GOOGLE_APPLICATION_CREDENTIALS"]
SECRETS_PATH = os.environ["SECRETS_PATH"]
BUCKET_NAME = "fionaa-customer-assets"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = client.bucket(BUCKET_NAME)


In [4]:
# List all blobs/files in the bucket
blobs = list(bucket.list_blobs())

# Print the file names
print("Files in bucket '{}':".format(bucket))
for blob in blobs:
    print(blob.name)

# Example: Read the contents of a file (e.g., PDF as bytes)
file_name = "data/5573 - Draft Accounts.pdf"
blob = bucket.blob(file_name)
file_bytes = blob.download_as_bytes()
print(f"Downloaded {file_name}, {len(file_bytes)} bytes")




Files in bucket '<Bucket: fionaa-customer-assets>':
data/5573 - Draft Accounts.pdf
data/fd statement 2143 20240622.pdf
data/fd statement 2383 20241130 (1).pdf
data/fd statement 2383 20241130.pdf
Downloaded data/5573 - Draft Accounts.pdf, 134452 bytes


In [5]:
# This is for if we want to turn Gmail into a tool, not just ingesting emails.
# Might want to do this for the response agent when we need more details
# from the user such as missing information or clarification on the users request.

from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)
from langchain_google_community import GmailToolkit



# Use the same scopes that were used when creating the token
# These must match the scopes in token.json
SCOPES = [
    'https://www.googleapis.com/auth/gmail.modify',
    'https://www.googleapis.com/auth/calendar'
]

credentials = get_gmail_credentials(
    token_file=SECRETS_PATH+"token.json",
    scopes=SCOPES,
    client_sercret_file=SECRETS_PATH+"secrets.json",  # Note: setup script uses secrets.json
)
api_resource = build_resource_service(credentials=credentials)
toolkit = GmailToolkit(api_resource=api_resource)

/var/folders/4b/dthvd31s2jnbxsmqfg30w0w00000gp/T/ipykernel_69654/3156223744.py:20: DeprecationWarning: get_gmail_credentials is deprecated and will be removed in a future version.Use get_google_credentials instead.
  credentials = get_gmail_credentials(
/var/folders/4b/dthvd31s2jnbxsmqfg30w0w00000gp/T/ipykernel_69654/3156223744.py:25: DeprecationWarning: build_resource_service is deprecated and will be removed in a future version.Use build_gmail_service instead.
  api_resource = build_resource_service(credentials=credentials)


In [33]:
tools = toolkit.get_tools()
tools

[GmailCreateDraft(api_resource=<googleapiclient.discovery.Resource object at 0x120e76450>),
 GmailSendMessage(api_resource=<googleapiclient.discovery.Resource object at 0x120e76450>),
 GmailSearch(api_resource=<googleapiclient.discovery.Resource object at 0x120e76450>),
 GmailGetMessage(api_resource=<googleapiclient.discovery.Resource object at 0x120e76450>),
 GmailGetThread(api_resource=<googleapiclient.discovery.Resource object at 0x120e76450>)]

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [27]:
# Use LangGraph instead of create_agent (which has compatibility issues)
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import Literal

# Define state for the agent
class AgentState(MessagesState):
    pass

# Get tools from toolkit and create a tools_by_name dict
tools = toolkit.get_tools()
tools_by_name = {tool.name: tool for tool in tools}

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Define LLM node
def llm_call(state: AgentState):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            llm_with_tools.invoke(state["messages"])
        ]
    }

# Define tool execution node
def tool_node(state: AgentState):
    """Execute tool calls"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append({
            "role": "tool", 
            "content": str(observation), 
            "tool_call_id": tool_call["id"]
        })
    return {"messages": result}

# Conditional edge function
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """Route to tools if there are tool calls, otherwise end"""
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# Build the agent graph
agent_builder = StateGraph(AgentState)
agent_builder.add_node("llm", llm_call)
agent_builder.add_node("tools", tool_node)
agent_builder.add_edge(START, "llm")
agent_builder.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tools": "tools",
        "__end__": END,
    },
)
agent_builder.add_edge("tools", "llm")

# Compile the agent
agent_executor = agent_builder.compile()

In [ ]:
from langchain_core.messages import HumanMessage

example_query = "Draft an email to fake@fake.com thanking them for coffee."

# Invoke the agent (LangGraph uses invoke, not stream with stream_mode)
result = agent_executor.invoke({
    "messages": [HumanMessage(content=example_query)]
})

# Print the final messages
for message in result["messages"]:
    print(f"{message.__class__.__name__}: {message.content}")
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"  Tool calls: {message.tool_calls}")

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


HumanMessage: Draft an email to fake@fake.com thanking them for coffee.
AIMessage: 
  Tool calls: [{'name': 'create_gmail_draft', 'args': {'message': 'Thank you for the coffee! I really enjoyed our time together and appreciate the opportunity to catch up. Looking forward to our next meeting!', 'to': ['fake@fake.com'], 'subject': 'Thank You for the Coffee!'}, 'id': 'call_1N74qrARDFih2YfiFXdzNcvw', 'type': 'tool_call'}]
ToolMessage: Draft created. Draft Id: r5098619186479425392
AIMessage: I've drafted an email thanking them for coffee. If you need any changes or want to send it, just let me know!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

In [29]:
example_query = "get the emails received in the past 24 hours."

# Invoke the agent (LangGraph uses invoke, not stream with stream_mode)
result = agent_executor.invoke({
    "messages": [HumanMessage(content=example_query)]
})

# Print the final messages
for message in result["messages"]:
    print(f"{message.__class__.__name__}: {message.content}")
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"  Tool calls: {message.tool_calls}")

HumanMessage: get the emails received in the past 24 hours.
AIMessage: 
  Tool calls: [{'name': 'search_gmail', 'args': {'query': 'after:2023/10/25', 'resource': 'messages'}, 'id': 'call_2NJaUFYyWw22qZ8bUsWXlyej', 'type': 'tool_call'}]
ToolMessage: [{'id': '19b32b0d3e8285b3', 'threadId': '19b32b0d3e8285b3', 'snippet': 'Thank you for the coffee! I really enjoyed our time together and appreciate the opportunity to catch up. Looking forward to our next meeting!', 'body': 'Thank you for the coffee! I really enjoyed our time together and appreciate the opportunity to catch up. Looking forward to our next meeting!\n', 'subject': 'Thank You for the Coffee!', 'sender': 'stevejgoodman@gmail.com'}, {'id': '19b326658d231337', 'threadId': '19b326658d231337', 'snippet': 'Take a look at what you accomplished this year, diving into your personal journey of growth and the remarkable progress you&#39;ve made along the way! LogoFiles_DeepLearning_PrimaryLogo-Aug-12-2021-05-', 'body': 'Take a look at wha